> Notebook-friendly copy of `part-I/1.2-data-structures-and-control-flow-environmental-solutions.ipynb`, generated by `tools/make_live.py`. Edit the book notebook, not this file.

In [ ]:
# --- environment setup (generated, not part of the lesson) ---
# Colab and Kaggle do not ship every package this notebook imports.
# This is a no-op in an environment that is already set up.
import importlib.util
import subprocess
import sys

for module, package in {"pooch": "pooch"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

# Environmental Solutions

**ℹ️ Reference solutions**

Worked solutions for the real-dataset exercises in [1.2-data-structures-and-control-flow-environmental-exercises.ipynb](1.2-data-structures-and-control-flow-environmental-exercises.ipynb).

## Exercise 1: A year of station data, continued

Exercises 1 and 2 are the main exercises of this subchapter.

In subchapter 1.1 you fetched this file, worked out its structure, and converted a single
record to °C. You did not yet have the tools to handle all of it at once — now you do.

The goal: turn a year of daily maximum temperatures into a monthly summary, and write that
summary out as a new file. The input file has the same three columns as in Exercise 1 of the
previous subchapter's environmental exercises: the station code, the day as `dd.mm.yy`, and the daily maximum temperature
in degrees Fahrenheit. Your monthly summary will have its own columns: `month`, `n_obs`
(the number of observations), and `mean_temp_celsius`.

In [ ]:
# Pre-supplied: download the data file and cache it locally.
# You do not need to understand this cell yet — fetching data is covered in the
# reproducible-data-pipelines bonus subchapter.
import pooch

data_file = pooch.retrieve(
    url="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/data/part-I/station_iib_daily_max_temp_2022.csv",
    known_hash="sha256:034755fb289b4e157e5d029995481786a359eb25a80df62c09ee83d063013144",
    fname="station_iib_daily_max_temp_2022.csv",
    path=pooch.os_cache("mlees"),
)
print("data cached at:", data_file)

**Step 1.** Write a function `fahrenheit_to_celsius(temp_fahrenheit)` that returns the
temperature in °C. Test it on the freezing point of water (32 °F should give 0.0 °C) and on
212 °F. A one-line docstring is optional here (see Exercise 11 of the practice exercises).

You wrote this conversion inline in 1.1. Steps 2 and 4 both need it again; with a function,
they call the same tested code instead of repeating the arithmetic.

In [ ]:
def fahrenheit_to_celsius(temp_fahrenheit):
    return (temp_fahrenheit - 32.0) * 5.0 / 9.0


print(fahrenheit_to_celsius(32.0))    # freezing point of water: 0.0
print(fahrenheit_to_celsius(212.0))   # boiling point of water: 100.0

**Step 2.** Open the file, read all lines, and loop over the records (skipping the header).
For each one, extract the month and the temperature in °C. Print the first three as a check.

Sixteen days in this file have an empty temperature field — a real gap in the station's record,
late October to mid-November. `float("")` raises `ValueError`, so your loop has to decide what to
do with those days before it can get to the end of the file. Skip them, and count how many you
skipped. After the loop, print how many records you converted and how many you skipped; the two
should add up to 365.

In [ ]:
from pathlib import Path

In [ ]:
with Path(data_file).open("r", encoding="utf-8") as fhandler:
    lines = fhandler.readlines()

In [ ]:
# 16 days in this file have an empty temperature field -- a real sensor gap. float("")
# raises ValueError, so the loop has to decide what to do with them; here it skips them
# and counts them, and step 7 turns that into a general guard.
n_converted = 0
n_skipped = 0
for line in lines[1:]:                      # skip the header
    fields = line.strip().split(",")
    day = fields[1]                         # dd.mm.yy
    month = day.split(".")[1]
    if fields[2] == "":
        n_skipped += 1
        continue
    temp_celsius = fahrenheit_to_celsius(float(fields[2]))
    if n_converted < 3:
        print(f"month {month}: {temp_celsius:.1f} °C")
    n_converted += 1

In [ ]:
print("converted:", n_converted, "| skipped:", n_skipped)    # 349 and 16, which add up to 365

**Step 3.** Build a dictionary mapping each month to the number of observations in it.

You do not know in advance which months are present, so the dictionary has to grow as you go.
Keep skipping the days with no reading: a day the sensor did not record is not an observation, so
October and November should come out short of their full length.

In [ ]:
counts = {}
for line in lines[1:]:
    fields = line.strip().split(",")
    if fields[2] == "":          # a day with no reading is not an observation
        continue
    month = fields[1].split(".")[1]
    counts[month] = counts.get(month, 0) + 1

print(counts)

**Step 4.** Build a second dictionary mapping each month to its **mean** daily maximum
temperature in °C. Print it with each mean rounded to two decimals.

In [ ]:
counts = {}
totals = {}
for line in lines[1:]:
    fields = line.strip().split(",")
    if fields[2] == "":
        continue
    month = fields[1].split(".")[1]
    temp_celsius = fahrenheit_to_celsius(float(fields[2]))
    counts[month] = counts.get(month, 0) + 1
    totals[month] = totals.get(month, 0.0) + temp_celsius

In [ ]:
means = {}
for month, total_celsius in totals.items():
    means[month] = total_celsius / counts[month]

In [ ]:
for month, mean_celsius in means.items():
    print(month, round(mean_celsius, 2))

**Step 5.** Find the warmest month and print it with its mean temperature.

Do this with a loop and a comparison, keeping track of the best value seen so far, as you did for
the warmest day in Exercise 6 of the practice exercises. A dictionary is looked up by key, not by position, so there is no
`[0]` to start from: start from `None` instead.

In [ ]:
warmest_month = None
warmest_mean = None
for month, mean_celsius in means.items():
    if warmest_mean is None:          # the first month seen is the best so far
        warmest_month = month
        warmest_mean = mean_celsius
    elif mean_celsius > warmest_mean:
        warmest_month = month
        warmest_mean = mean_celsius

print(f"warmest month: {warmest_month} at {warmest_mean:.2f} °C")

**Step 6.** Write your monthly summary to `_files/monthly_means.csv`, with the header
`month,n_obs,mean_temp_celsius` and one line per month, creating the `_files` folder first as you
did in 1.1. Then read the file back and print it, to confirm it says what you meant it to say.

In [ ]:
out_path = Path("_files") / "monthly_means.csv"
out_path.parent.mkdir(exist_ok=True)        # _files/ is not there on a fresh clone or in Colab

with out_path.open("w", encoding="utf-8") as fhandler:
    fhandler.write("month,n_obs,mean_temp_celsius\n")
    for month, mean_celsius in means.items():
        fhandler.write(f"{month},{counts[month]},{mean_celsius:.2f}\n")

with out_path.open("r", encoding="utf-8") as fhandler:
    print(fhandler.read())

**Step 7 (optional).** This step uses `try`/`except` from the going-deeper box in the lecture. Real files contain junk, and you have already met one kind of it — the empty
temperature field. Suppose some lines are truncated as well, or hold a temperature that is not a
number at all. Modify your reading loop so that it skips any line that does not have three fields
or whose temperature will not convert to a number, counting how many lines it skipped.

Test it on this list before applying it to the real file:

```python
test_lines = [
    "IIB,01.01.22,30.2",
    "IIB,02.01.22",
    "IIB,03.01.22,notanumber",
    "IIB,04.01.22,33.4",
]
```

You should keep 2 and skip 2.

In [ ]:
test_lines = [
    "IIB,01.01.22,30.2",
    "IIB,02.01.22",
    "IIB,03.01.22,notanumber",
    "IIB,04.01.22,33.4",
]

In [ ]:
n_kept = 0
n_skipped = 0
for line in test_lines:
    fields = line.strip().split(",")
    if len(fields) != 3:
        n_skipped += 1
        continue
    try:
        temp_celsius = fahrenheit_to_celsius(float(fields[2]))
    except ValueError:
        n_skipped += 1
        continue
    n_kept += 1

In [ ]:
print(f"test lines: kept {n_kept}, skipped {n_skipped}")

In [ ]:
# the same guard applied to the real file
n_kept = 0
n_skipped = 0
for line in lines[1:]:
    fields = line.strip().split(",")
    if len(fields) != 3:
        n_skipped += 1
        continue
    try:
        temp_celsius = fahrenheit_to_celsius(float(fields[2]))
    except ValueError:
        n_skipped += 1
        continue
    n_kept += 1

In [ ]:
print(f"real file: kept {n_kept}, skipped {n_skipped}")

## Exercise 2: The solar system

<img src="https://raw.githubusercontent.com/gse-unil/2026_MLEES_book/main/part-I/_static/nasa-solar-system-montage.jpg" alt="The eight planets and the Moon photographed by spacecraft, arranged diagonally against black" width="100%">

<em>The eight planets and the Moon, as photographed by Mariner 10, Magellan, Galileo, Mars Global
Surveyor, Cassini and the Voyager spacecraft. The inner bodies are roughly to scale with each other, as are the outer ones, but the
two groups are not to scale with one another. Image
<a href="https://www.jpl.nasa.gov/images/pia03153-solar-system-montage-high-resolution-2001-version/">NASA/JPL (PIA03153)</a>, public domain.</em>


This is the long exercise for this subchapter, and the only one in it whose data is not
environmental. Eight planets and their masses are few enough to type by hand, and familiar enough
that a wrong answer is easy to spot, which makes them a good first test of lists, dictionaries and
functions together.

Work through sections A to C in order: section C writes functions against the dictionary you
build in section B. You will pick the exercise up again in 1.7, once classes are available, and
turn the dictionary into a `Planet` type.

The masses come from NASA's
[planetary fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/). Use units of
10^24 kg throughout, so that Earth is 5.97 and Jupiter is 1898 — the variable names below carry
that unit, which is what makes a bare number like `1898` readable three cells later.

### A: lists and loops

**Q1) Create a list with the names of every planet in the solar system, in order.**

Name it `planets` and use lowercase names, so it starts `["mercury", "venus", ...]`.

In [ ]:
planets = ["mercury", "venus", "earth", "mars", "jupiter", "saturn", "uranus", "neptune"]
print(planets)

**Q2) Have Python tell you how many planets there are by examining your list.**

You should get 8. Do not type the number — read it off the list.

In [ ]:
# len() reads the length off the list, so the answer stays right if the list changes
print(len(planets))

**Q3) Use slicing to display the first four planets — the rocky ones.**

In [ ]:
# [:4] is the first four elements -- the rocky planets
print(planets[:4])

**Q4) Iterate through your planets and print the planet name only if it ends in "s".**

You should see `venus`, `mars` and `uranus`, and nothing else.

Hint: the last letter of a name is `planet[-1]`. Use `continue` for the names that do not match,
so the loop body says explicitly what it skips.

In [ ]:
for planet in planets:
    if planet[-1] != "s":
        continue          # not a match: skip to the next name
    print(planet)

### B: dictionary

**Q5) Create a dictionary that maps each planet name to its mass.**

Call it `planet_masses_1e24kg`, and read the values off the
[NASA fact sheet](https://nssdc.gsfc.nasa.gov/planetary/factsheet/) in units of 10^24 kg. Every
value has to be in that one unit: entered as 1.898 (in 10^27 kg), Jupiter would come out 1000
times too light in every comparison and ratio below.

In [ ]:
# masses in units of 1e24 kg, from the NASA planetary fact sheet
planet_masses_1e24kg = {
    "mercury": 0.330,
    "venus": 4.87,
    "earth": 5.97,
    "mars": 0.642,
    "jupiter": 1898.0,
    "saturn": 568.0,
    "uranus": 86.8,
    "neptune": 102.0,
}
print(planet_masses_1e24kg)

**Q6) Use your dictionary to look up Earth's mass.**

In [ ]:
print(planet_masses_1e24kg["earth"])      # 5.97

**Q7) Loop through the dictionary and build a list of every planet heavier than
100 x 10^24 kg.**

You should end up with `["jupiter", "saturn", "neptune"]` — uranus, at 86.8, does not make it.

Hint: `.items()` gives you the name and the mass together on each pass, as in 1.2's lecture.

In [ ]:
heavy_planets = []
for name, mass_1e24kg in planet_masses_1e24kg.items():
    if mass_1e24kg > 100.0:
        heavy_planets.append(name)
print(heavy_planets)                      # ['jupiter', 'saturn', 'neptune']

# uranus is 86.8, so it misses the threshold by a wide margin -- worth checking by eye
print(planet_masses_1e24kg["uranus"])

**Q8) Add Pluto to your dictionary.**

Its mass is 0.0130 x 10^24 kg. Print the names afterwards, looping with `.items()`, to confirm it
landed.

Pluto has not been a planet since 2006. Adding it still takes one assignment, and none of the code
that reads the other eight entries has to change.

In [ ]:
planet_masses_1e24kg["pluto"] = 0.0130
for name, mass_1e24kg in planet_masses_1e24kg.items():
    print(name)

### C: functions

**Q9) Write a function that converts a planetary mass to Earth masses.**

Jupiter is about 1898 x 10^24 kg, which is about 318 Earth masses:

$$M_{\text{Jupiter}} \approx 1898 \times 10^{24}\,\text{kg} \approx 318\,M_{\text{Earth}}$$

Call it `to_earth_masses(mass_1e24kg)` and check it against Jupiter — you should get
317.92294807370183.

In [ ]:
def to_earth_masses(mass_1e24kg):
    return mass_1e24kg / planet_masses_1e24kg["earth"]

print(to_earth_masses(planet_masses_1e24kg["jupiter"]))   # 317.92294807370183

**Q10) Now write a single function that converts to either Earth masses or Jupiter
masses, depending on a keyword argument.**

Call it `to_planet_masses(mass_1e24kg, reference="jupiter")`. Converting Jupiter's own mass should
give 317.923 with `reference="earth"` and exactly 1.0 with `reference="jupiter"`.

With the default argument, one function covers both conversions. Jupiter in Jupiter masses has
to come out as exactly 1.0; any other value means the function divides by the wrong mass.

In [ ]:
def to_planet_masses(mass_1e24kg, reference="jupiter"):
    return mass_1e24kg / planet_masses_1e24kg[reference]

print(to_planet_masses(planet_masses_1e24kg["jupiter"], reference="earth"))   # 317.92294807370183
print(to_planet_masses(planet_masses_1e24kg["jupiter"]))                      # 1.0, the default reference

**Q11) Write a function that returns two values.**

`mass_in_two_units(mass_1e24kg, reference)` should return the mass in Earth masses *and* in masses
of whatever reference planet you pass. Test it on Jupiter with `reference="mars"`.

Hint: `return a, b` returns a tuple, and `first, second = f(...)` unpacks it — both from 1.2's
lecture.

In [ ]:
def mass_in_two_units(mass_1e24kg, reference):
    in_earth_masses = mass_1e24kg / planet_masses_1e24kg["earth"]
    in_reference_masses = mass_1e24kg / planet_masses_1e24kg[reference]
    return in_earth_masses, in_reference_masses

in_earth, in_mars = mass_in_two_units(planet_masses_1e24kg["jupiter"], reference="mars")
print(in_earth, in_mars)      # 317.92294807370183 2956.386292834891

**Bonus.** Convert Neptune's mass to Jupiter masses with your Q10 function. Then pass that result
to your Q11 function with `reference="jupiter"`, and compare the second value it returns with the
Q10 result. Are they the same? If not, work out which function is being given a number in the
wrong unit, and what it should have been given instead — this is the mistake the `_1e24kg` suffix
exists to prevent.

In [ ]:
neptune_in_jupiter_masses = to_planet_masses(planet_masses_1e24kg["neptune"])
print(neptune_in_jupiter_masses)                    # 0.05374077976817703

# passing the Q10 result into Q11 does NOT give it back
in_earth, in_jupiter = mass_in_two_units(neptune_in_jupiter_masses, reference="jupiter")
print(in_jupiter)                                   # 2.831442558913437e-05

# Q11 expects a mass in 1e24 kg, so give it Neptune's mass itself
in_earth, in_jupiter = mass_in_two_units(planet_masses_1e24kg["neptune"], reference="jupiter")
print(in_jupiter)                                   # 0.05374077976817703, the Q10 result

# Q10 returned a number in Jupiter masses; Q11 expects one in 1e24 kg. The two functions
# disagree about the unit of their input, and nothing in the code says so -- which is why
# the suffix belongs in the variable name rather than in a comment.

**ℹ️ What you just did**

You built a list and read its length rather than counting it, sliced a subset out of it, and
looped over it with a condition. You built a dictionary keyed by name, looked a value up, filtered
it into a new list, and extended it. Then you wrote three functions over that dictionary: one
fixed, one with a default argument that collapses two functions into one, and one returning a pair.

That is the whole of this subchapter applied to one dataset. In 1.7 you will come back and replace
the dictionary with a class, so that a planet's name and mass travel together as one object
instead of as a key and a value that happen to line up.